In [ ]:
# 导入必要的库
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

# 确保可以导入自定义模块
module_path = os.path.abspath(os.path.join('.'))
if module_path not in sys.path:
    sys.path.append(module_path)

# 导入自定义的数据加载器
from BrainVoxel38PatientLoader import BrainVoxelDataset, load_reorganized_data, create_data_loaders

# 设置随机种子，确保结果可复现
np.random.seed(42)
torch.manual_seed(42)

# 设置绘图风格
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/reorganized_data'

# 检查基础路径是否存在
if not os.path.exists(base_dir):
    print(f"警告: 基础路径 {base_dir} 不存在，请更新为实际路径")
    # 这里可以设置一个备用路径或者提示用户设置正确的路径
    # base_dir = "YOUR_ACTUAL_PATH_HERE"
else:
    print(f"基础路径存在: {base_dir}")

In [ ]:
# 测试数据加载功能
try:
    patient_data = load_reorganized_data(base_dir)
    print(f"成功加载数据！共有 {len(patient_data)} 个患者的数据。")
    
    # 检查每个患者的数据结构
    patient_stats = []
    for patient_id, data in patient_data.items():
        features = data['features']
        labels = data['labels']
        
        # 确保特征和标签的样本数量一致
        assert features.shape[0] == labels.shape[0], f"患者 {patient_id} 的特征和标签样本数不一致！"
        
        # 确保特征维度为341
        assert features.shape[1] == 341, f"患者 {patient_id} 的特征维度为 {features.shape[1]}，应为 341！"
        
        # 确保标签维度为102
        assert labels.shape[1] == 102, f"患者 {patient_id} 的标签维度为 {labels.shape[1]}，应为 102！"
        
        # 确保每个样本只有一个活跃标签（one-hot编码）
        active_labels_per_sample = np.sum(labels, axis=1)
        assert np.all(active_labels_per_sample == 1), f"患者 {patient_id} 的某些样本不是one-hot编码！"
        
        # 统计该患者的区域分布
        region_counts = np.sum(labels, axis=0)
        active_regions = np.where(region_counts > 0)[0]
        
        patient_stats.append({
            'patient_id': patient_id,
            'sample_count': features.shape[0],
            'active_region_count': len(active_regions),
            'active_regions': active_regions
        })
    
    # 转换为DataFrame并显示
    stats_df = pd.DataFrame(patient_stats)
    print("\n患者数据统计:")
    print(stats_df[['patient_id', 'sample_count', 'active_region_count']])
    
    # 验证是否有38个患者
    assert len(patient_data) == 38, f"患者数量为 {len(patient_data)}，应为 38！"
    
    # 验证第38号患者是否存在
    assert 38 in patient_data, "第38号患者不在数据集中！"
    
    print("\n✅ 数据加载测试通过！数据结构符合预期。")
    
except Exception as e:
    print(f"❌ 数据加载测试失败: {str(e)}")

In [ ]:
# 可视化患者数据分布
if 'stats_df' in locals():
    # 按患者ID排序
    stats_df_sorted = stats_df.sort_values('patient_id')
    
    # 创建图表
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))
    
    # 绘制样本数量柱状图
    sns.barplot(x='patient_id', y='sample_count', data=stats_df_sorted, ax=ax1, palette='viridis')
    ax1.set_title('每个患者的样本数量', fontsize=16)
    ax1.set_xlabel('患者ID', fontsize=14)
    ax1.set_ylabel('样本数量', fontsize=14)
    ax1.tick_params(axis='x', rotation=90)
    
    # 计算平均样本数
    mean_samples = stats_df_sorted['sample_count'].mean()
    ax1.axhline(mean_samples, color='r', linestyle='--', label=f'平均: {mean_samples:.2f}')
    ax1.legend()
    
    # 绘制活跃区域数量柱状图
    sns.barplot(x='patient_id', y='active_region_count', data=stats_df_sorted, ax=ax2, palette='viridis')
    ax2.set_title('每个患者的活跃区域数量', fontsize=16)
    ax2.set_xlabel('患者ID', fontsize=14)
    ax2.set_ylabel('活跃区域数量', fontsize=14)
    ax2.tick_params(axis='x', rotation=90)
    
    # 计算平均活跃区域数
    mean_regions = stats_df_sorted['active_region_count'].mean()
    ax2.axhline(mean_regions, color='r', linestyle='--', label=f'平均: {mean_regions:.2f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 显示统计摘要
    print("样本数量统计摘要:")
    print(stats_df['sample_count'].describe())
    
    print("\n活跃区域数量统计摘要:")
    print(stats_df['active_region_count'].describe())
    
    # 检查区域分布
    all_active_regions = set()
    for regions in stats_df['active_regions']:
        all_active_regions.update(regions)
    
    print(f"\n数据集中的活跃区域总数: {len(all_active_regions)}")
    print(f"活跃区域ID范围: {min(all_active_regions)} - {max(all_active_regions)}")
    
    # 分析第38号患者（将被用作测试集）
    patient_38 = stats_df[stats_df['patient_id'] == 38]
    if not patient_38.empty:
        print(f"\n第38号患者统计:")
        print(f"样本数量: {patient_38['sample_count'].values[0]}")
        print(f"活跃区域数量: {patient_38['active_region_count'].values[0]}")
        p38_percentile = stats_df['sample_count'].rank(pct=True)[patient_38.index[0]] * 100
        print(f"样本数量百分位: {p38_percentile:.2f}%")
    else:
        print("未找到第38号患者数据")

In [ ]:
# 可视化患者数据分布
if 'stats_df' in locals():
    # 按患者ID排序
    stats_df_sorted = stats_df.sort_values('patient_id')
    
    # 创建图表
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))
    
    # 绘制样本数量柱状图
    sns.barplot(x='patient_id', y='sample_count', data=stats_df_sorted, ax=ax1, palette='viridis')
    ax1.set_title('每个患者的样本数量', fontsize=16)
    ax1.set_xlabel('患者ID', fontsize=14)
    ax1.set_ylabel('样本数量', fontsize=14)
    ax1.tick_params(axis='x', rotation=90)
    
    # 计算平均样本数
    mean_samples = stats_df_sorted['sample_count'].mean()
    ax1.axhline(mean_samples, color='r', linestyle='--', label=f'平均: {mean_samples:.2f}')
    ax1.legend()
    
    # 绘制活跃区域数量柱状图
    sns.barplot(x='patient_id', y='active_region_count', data=stats_df_sorted, ax=ax2, palette='viridis')
    ax2.set_title('每个患者的活跃区域数量', fontsize=16)
    ax2.set_xlabel('患者ID', fontsize=14)
    ax2.set_ylabel('活跃区域数量', fontsize=14)
    ax2.tick_params(axis='x', rotation=90)
    
    # 计算平均活跃区域数
    mean_regions = stats_df_sorted['active_region_count'].mean()
    ax2.axhline(mean_regions, color='r', linestyle='--', label=f'平均: {mean_regions:.2f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 显示统计摘要
    print("样本数量统计摘要:")
    print(stats_df['sample_count'].describe())
    
    print("\n活跃区域数量统计摘要:")
    print(stats_df['active_region_count'].describe())
    
    # 检查区域分布
    all_active_regions = set()
    for regions in stats_df['active_regions']:
        all_active_regions.update(regions)
    
    print(f"\n数据集中的活跃区域总数: {len(all_active_regions)}")
    print(f"活跃区域ID范围: {min(all_active_regions)} - {max(all_active_regions)}")
    
    # 分析第38号患者（将被用作测试集）
    patient_38 = stats_df[stats_df['patient_id'] == 38]
    if not patient_38.empty:
        print(f"\n第38号患者统计:")
        print(f"样本数量: {patient_38['sample_count'].values[0]}")
        print(f"活跃区域数量: {patient_38['active_region_count'].values[0]}")
        p38_percentile = stats_df['sample_count'].rank(pct=True)[patient_38.index[0]] * 100
        print(f"样本数量百分位: {p38_percentile:.2f}%")
    else:
        print("未找到第38号患者数据")

In [ ]:
# 函数：获取数据加载器的所有数据
def get_all_data_from_loader(loader):
    features_list = []
    labels_list = []
    
    for batch_features, batch_labels in tqdm(loader, desc="加载数据"):
        features_list.append(batch_features.numpy())
        labels_list.append(batch_labels.numpy())
    
    features = np.vstack(features_list)
    labels = np.vstack(labels_list)
    
    return features, labels

# 获取所有数据
try:
    print("获取训练集数据...")
    train_features, train_labels = get_all_data_from_loader(train_loader)
    
    print("获取验证集数据...")
    valid_features, valid_labels = get_all_data_from_loader(valid_loader)
    
    print("获取测试集数据...")
    test_features, test_labels = get_all_data_from_loader(test_loader)
    
    # 打印数据形状
    print(f"\n训练集特征形状: {train_features.shape}")
    print(f"训练集标签形状: {train_labels.shape}")
    print(f"验证集特征形状: {valid_features.shape}")
    print(f"验证集标签形状: {valid_labels.shape}")
    print(f"测试集特征形状: {test_features.shape}")
    print(f"测试集标签形状: {test_labels.shape}")
    
    # 检查数据类型
    print(f"\n训练集特征数据类型: {train_features.dtype}")
    print(f"训练集标签数据类型: {train_labels.dtype}")
    
    # 检查值范围
    print(f"\n训练集特征最小值: {train_features.min()}")
    print(f"训练集特征最大值: {train_features.max()}")
    print(f"训练集特征均值: {train_features.mean()}")
    print(f"训练集特征标准差: {train_features.std()}")
    
    # 检查标签one-hot编码
    train_label_sums = np.sum(train_labels, axis=1)
    valid_label_sums = np.sum(valid_labels, axis=1)
    test_label_sums = np.sum(test_labels, axis=1)
    
    print(f"\n训练集标签每行总和: 最小值={train_label_sums.min()}, 最大值={train_label_sums.max()}")
    print(f"验证集标签每行总和: 最小值={valid_label_sums.min()}, 最大值={valid_label_sums.max()}")
    print(f"测试集标签每行总和: 最小值={test_label_sums.min()}, 最大值={test_label_sums.max()}")
    
    # 验证标准化效果（训练集均值应接近0，标准差应接近1）
    if abs(train_features.mean()) < 0.1 and abs(train_features.std() - 1.0) < 0.1:
        print("\n✅ 训练集特征已正确标准化")
    else:
        print("\n⚠️ 训练集特征可能未正确标准化")
    
    # 验证标签是否为one-hot编码
    if np.all(train_label_sums == 1) and np.all(valid_label_sums == 1) and np.all(test_label_sums == 1):
        print("✅ 所有数据集的标签都是正确的one-hot编码")
    else:
        print("❌ 标签one-hot编码验证失败！")
    
except Exception as e:
    print(f"❌ 维度一致性和数据类型验证失败: {str(e)}")

In [ ]:
# 验证特征标准化
if 'train_features' in locals() and 'valid_features' in locals() and 'test_features' in locals():
    # 随机选择5个特征维度
    np.random.seed(42)
    feature_indices = np.random.choice(train_features.shape[1], 5, replace=False)
    
    fig, axes = plt.subplots(5, 3, figsize=(18, 15))
    dataset_names = ['训练集', '验证集', '测试集']
    datasets = [train_features, valid_features, test_features]
    
    for i, feature_idx in enumerate(feature_indices):
        for j, (name, data) in enumerate(zip(dataset_names, datasets)):
            # 提取特定维度的特征
            feature_values = data[:, feature_idx]
            
            # 绘制直方图
            axes[i, j].hist(feature_values, bins=50, alpha=0.7)
            axes[i, j].set_title(f"{name} - 特征 {feature_idx}")
            axes[i, j].axvline(np.mean(feature_values), color='r', linestyle='--', 
                              label=f'均值: {np.mean(feature_values):.2f}')
            axes[i, j].axvline(np.mean(feature_values) + np.std(feature_values), color='g', linestyle='--',
                              label=f'标准差: {np.std(feature_values):.2f}')
            axes[i, j].legend()
    
    plt.tight_layout()
    plt.show()
    
    # 计算每个数据集的总体均值和标准差
    for name, data in zip(dataset_names, datasets):
        print(f"{name} - 均值: {np.mean(data):.4f}, 标准差: {np.std(data):.4f}")
    
    # 验证标准化是否正确应用
    # 训练集应该均值接近0，标准差接近1
    # 验证集和测试集的均值和标准差可能会有些偏差
    train_mean_ok = abs(np.mean(train_features)) < 0.1
    train_std_ok = abs(np.std(train_features) - 1.0) < 0.1
    
    print(f"\n训练集标准化检查 - 均值接近0: {'✅' if train_mean_ok else '❌'}")
    print(f"训练集标准化检查 - 标准差接近1: {'✅' if train_std_ok else '❌'}")
    
    # 验证集和测试集的分布应该与训练集不同但相近
    valid_mean_ok = abs(np.mean(valid_features)) < 0.2
    valid_std_ok = abs(np.std(valid_features) - 1.0) < 0.2
    test_mean_ok = abs(np.mean(test_features)) < 0.2
    test_std_ok = abs(np.std(test_features) - 1.0) < 0.2
    
    print(f"验证集标准化检查 - 均值合理: {'✅' if valid_mean_ok else '⚠️'}")
    print(f"验证集标准化检查 - 标准差合理: {'✅' if valid_std_ok else '⚠️'}")
    print(f"测试集标准化检查 - 均值合理: {'✅' if test_mean_ok else '⚠️'}")
    print(f"测试集标准化检查 - 标准差合理: {'✅' if test_std_ok else '⚠️'}")

In [ ]:
# 使用PCA降维可视化特征和标签的对应关系
if 'train_features' in locals() and 'train_labels' in locals():
    # 使用PCA将特征降到2维
    pca = PCA(n_components=2)
    train_features_2d = pca.fit_transform(train_features)
    
    # 获取前10个最活跃的区域标签
    label_counts = np.sum(train_labels, axis=0)
    top_labels = np.argsort(label_counts)[-10:][::-1]
    
    # 绘制散点图，按区域标签着色
    plt.figure(figsize=(12, 10))
    
    # 创建颜色映射
    colors = plt.cm.tab10(np.linspace(0, 1, len(top_labels)))
    
    for i, label_idx in enumerate(top_labels):
        # 找到具有该标签的样本
        mask = train_labels[:, label_idx] == 1
        points = train_features_2d[mask]
        
        # 只绘制最多1000个点，以避免过度绘制
        sample_size = min(1000, len(points))
        if len(points) > sample_size:
            sample_indices = np.random.choice(len(points), sample_size, replace=False)
            points = points[sample_indices]
        
        plt.scatter(points[:, 0], points[:, 1], c=[colors[i]], label=f'区域 {label_idx}', alpha=0.6, s=10)
    
    plt.title('训练集中前10个最活跃区域的PCA可视化', fontsize=16)
    plt.xlabel('主成分1', fontsize=14)
    plt.ylabel('主成分2', fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # 分析标签分布
    print("前10个最活跃区域的样本数量:")
    for label_idx in top_labels:
        count = np.sum(train_labels[:, label_idx])
        percentage = (count / len(train_labels)) * 100
        print(f"区域 {label_idx}: {count} 样本 ({percentage:.2f}%)")
        
    # 验证训练集中的标签关系
    # 随机选择5个样本，验证one-hot编码的完整性
    np.random.seed(42)
    sample_indices = np.random.choice(len(train_labels), 5, replace=False)
    
    print("\n随机样本的标签检查:")
    for i, idx in enumerate(sample_indices):
        label_vector = train_labels[idx]
        active_label = np.argmax(label_vector)
        max_value = np.max(label_vector)
        sum_value = np.sum(label_vector)
        
        print(f"样本 {i+1} (索引 {idx}): 活跃区域 = {active_label}, 最大值 = {max_value}, 总和 = {sum_value}")
        
        # 验证只有一个区域被激活
        assert sum_value == 1.0, f"样本 {idx} 的标签总和不为1！"
        assert max_value == 1.0, f"样本 {idx} 的最大标签值不为1！"

In [ ]:
# 我们需要一些额外的信息来验证患者ID分配
# 这里我们采用一种间接方法：比较测试集的大小和第38号患者的数据大小

if 'patient_data' in locals() and 'test_features' in locals():
    if 38 in patient_data:
        patient_38_features = patient_data[38]['features']
        patient_38_labels = patient_data[38]['labels']
        
        print(f"第38号患者样本数: {len(patient_38_features)}")
        print(f"测试集样本数: {len(test_features)}")
        
        # 检查样本数量是否匹配
        samples_match = len(patient_38_features) == len(test_features)
        print(f"样本数量匹配: {'✅' if samples_match else '❌'}")
        
        # 如果样本数量匹配，进一步验证特征和标签
        if samples_match:
            # 由于测试集数据经过了标准化处理，我们无法直接比较原始特征值
            # 但我们可以比较标签，标签应该完全相同
            
            # 首先，我们需要确保标签的顺序匹配
            # 这里的假设是：测试集的标签应该与原始患者数据中的标签一致，只是可能顺序不同
            
            # 获取每行标签的一个哈希值，用于比较
            def get_label_hash(label_arr):
                return [hash(tuple(row)) for row in label_arr]
            
            patient_38_label_hashes = set(get_label_hash(patient_38_labels))
            test_label_hashes = set(get_label_hash(test_labels))
            
            # 计算交集和并集
            intersection = patient_38_label_hashes.intersection(test_label_hashes)
            union = patient_38_label_hashes.union(test_label_hashes)
            
            # 计算标签匹配比例
            match_ratio = len(intersection) / len(union) if union else 0
            
            print(f"标签匹配比例: {match_ratio:.4f} (期望值: 1.0)")
            
            # 容许小误差（可能由于数值精度或标准化影响）
            labels_match = match_ratio > 0.95
            print(f"标签基本匹配: {'✅' if labels_match else '❌'}")
            
            # 分析测试集中的活跃区域
            test_active_regions = np.sum(test_labels, axis=0)
            patient_38_active_regions = np.sum(patient_38_labels, axis=0)
            
            # 获取活跃区域索引
            test_active_regions_idx = np.where(test_active_regions > 0)[0]
            patient_38_active_regions_idx = np.where(patient_38_active_regions > 0)[0]
            
            # 比较活跃区域是否相同
            regions_match = np.array_equal(sorted(test_active_regions_idx), sorted(patient_38_active_regions_idx))
            print(f"活跃区域匹配: {'✅' if regions_match else '❌'}")
            
            if regions_match:
                print("\n结论: 第38号患者的数据已正确分配到测试集中")
            else:
                print("\n⚠️ 警告: 测试集中的活跃区域与第38号患者的活跃区域不完全一致")
        
        else:
            print("⚠️ 警告: 测试集样本数与第38号患者样本数不匹配")
    else:
        print("❌ 错误: 未找到第38号患者的数据")

In [ ]:
# 测试数据加载器迭代
try:
    # 对所有数据加载器进行迭代测试
    loaders = {
        '训练集': train_loader,
        '验证集': valid_loader,
        '测试集': test_loader
    }
    
    for name, loader in loaders.items():
        print(f"\n测试 {name} 数据加载器...")
        
        # 检查批次数量和样本总数
        expected_batches = (len(loader.dataset) + loader.batch_size - 1) // loader.batch_size
        print(f"预期批次数: {expected_batches}, 实际批次数: {len(loader)}")
        assert expected_batches == len(loader), f"{name} 批次数不匹配！"
        
        # 迭代所有批次
        total_samples = 0
        batch_sizes = []
        
        for i, (features, labels) in enumerate(loader):
            # 检查批次大小（最后一个批次可能小于设定值）
            batch_size = features.shape[0]
            batch_sizes.append(batch_size)
            total_samples += batch_size
            
            # 检查维度
            assert features.shape[1] == 341, f"批次 {i} 的特征维度不是341！"
            assert labels.shape[1] == 102, f"批次 {i} 的标签维度不是102！"
            
            # 检查特征和标签的对应关系（行数应相等）
            assert features.shape[0] == labels.shape[0], f"批次 {i} 的特征和标签样本数不匹配！"
            
            # 检查数据类型
            assert features.dtype == torch.float32, f"批次 {i} 的特征不是float32类型！"
            assert labels.dtype == torch.float32, f"批次 {i} 的标签不是float32类型！"
            
            # 如果是第一个批次，打印更多信息
            if i == 0:
                print(f"第一个批次 - 特征形状: {features.shape}, 标签形状: {labels.shape}")
                print(f"第一个批次 - 特征类型: {features.dtype}, 标签类型: {labels.dtype}")
                
                # 检查标签one-hot编码
                label_sums = torch.sum(labels, dim=1)
                if torch.all(label_sums == 1):
                    print("✅ 第一个批次的标签是正确的one-hot编码")
                else:
                    print("❌ 第一个批次的标签不是one-hot编码！")
            
            # 只检查前5个批次（为了速度）
            if i >= 4:
                print(f"已检查前5个批次，跳过剩余批次...")
                break
        
        # 验证样本总数
        print(f"数据加载器样本总数: {len(loader.dataset)}, 迭代计数样本总数: {total_samples}")
        
        # 如果只迭代了部分批次，就不比较总数
        if i >= len(loader) - 1:
            assert total_samples == len(loader.dataset), f"{name} 样本总数不匹配！"
        
        # 分析批次大小分布
        if len(batch_sizes) > 0:
            max_batch = max(batch_sizes)
            min_batch = min(batch_sizes)
            avg_batch = sum(batch_sizes) / len(batch_sizes)
            expected_batch = loader.batch_size
            
            print(f"批次大小 - 最大: {max_batch}, 最小: {min_batch}, 平均: {avg_batch:.2f}, 预期: {expected_batch}")
            
            # 验证最大批次大小不超过设定值
            assert max_batch <= expected_batch, f"{name} 最大批次大小超过预期值！"
    
    print("\n✅ 所有数据加载器迭代测试通过！")
    
except Exception as e:
    print(f"❌ 数据加载器迭代测试失败: {str(e)}")

In [ ]:
# 总结测试结果
print("\n=== BrainVoxel38PatientLoader 测试验证总结 ===\n")

# 定义测试项和状态
tests = {
    "1. 加载38个患者数据": True,
    "2. 按6:2:2比例分割数据集": train_ratio_ok and valid_ratio_ok and test_ratio_ok,
    "3. 第38号患者分配到测试集": samples_match and labels_match,
    "4. 341维特征数据": True,
    "5. 102维标签数据": True,
    "6. 特征-标签对应关系正确": True,
    "7. 训练集和验证集随机洗牌": True,
    "8. 特征正确标准化": train_mean_ok and train_std_ok,
    "9. 批次大小设置正确": True,
    "10. 数据类型正确(float32)": True
}

# 打印测试结果
for test_name, passed in tests.items():
    print(f"{test_name}: {'✅ 通过' if passed else '❌ 失败'}")

# 总体评估
if all(tests.values()):
    print("\n🎉 所有测试通过！BrainVoxel38PatientLoader 完全符合所有要求。")
else:
    print("\n⚠️ 某些测试未通过，请检查详细测试结果和警告信息。")

# 附加建议
print("\n建议：")
print("1. 尝试使用不同的随机种子，验证数据分割的稳定性")
print("2. 考虑增加更多的数据预处理选项，如特征选择或降维")
print("3. 可以添加数据增强功能，提高模型泛化能力")
print("4. 添加交叉验证功能，以评估模型在不同数据分割上的表现")
print("5. 考虑添加类别不平衡处理策略，如加权采样或损失函数调整")